# Kinematics Calculator
This notebook is used for deriving and calculating the rig's kinematics

## Deriving the symbolic transformations Base -> End of Beam

In [2]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib
import sympy as sp
import plotly.io as pio

# Declaring the symbols
d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg = sp.symbols(
    r'd1 theta_y theta_p a_b h_b c_{boff} c_{h} c_{boc} \beta_{cam}',
    real=True
)

# Defining the DH parameters (with additional offsets on the links)
# Please note two more translation parameters are added to the DH table to account for the offsets in the links 
# The columns means as follows: [link number, link length, link twist, link offset (3 axes) joint angle]
DH = sp.Matrix([[1, 0, 0, 0, 0, d1, theta_y], \
            [2, 0, sp.pi/2, 0, 0, 0, theta_p], \
            [3, a_b, 0, 0, h_b, 0, 0], \
            [4, 0, sp.pi*camera_angle_deg/180, -camera_beam_offset,camera_height, camera_offset_from_beam_centre, 0]])

DH


Matrix([
[1,   0,                  0,         0,     0,      d1, theta_y],
[2,   0,               pi/2,         0,     0,       0, theta_p],
[3, a_b,                  0,         0,   h_b,       0,       0],
[4,   0, pi*\beta_{cam}/180, -c_{boff}, c_{h}, c_{boc},       0]])

In [4]:
# Calculating the individual transformation matrices using the DH parameters
N = len(DH[:, 0])

def create_transormation_string(superscript, subscript):
    return sp.symbols("{}^{" + str(superscript) + "}_" + str(subscript) + "T")


Full_forward_kinematics = sp.Matrix([[1, 0, 0, 0],\
                                  [0, 1, 0, 0],\
                                  [0, 0, 1, 0],
                                  [0, 0, 0, 1]])

Individual_transformation_matrices = []

for i in range(N):
    # Extracting parameters from the current link
    m = DH[i, 0]
    a_mm1 = DH[i, 1] #mm1 means m minus 1
    alpha_mm1 = DH[i, 2]
    tm = DH[i, 6]
    translate_i = DH[i, 3]
    translate_j = DH[i, 4]
    translate_k = DH[i, 5]



    print(f"=== LINK {m} ===")

    T_mm1_R = sp.Matrix([[1, 0, 0, 0], \
                         [0, sp.cos(alpha_mm1), -sp.sin(alpha_mm1), 0], \
                         [0, sp.sin(alpha_mm1), sp.cos(alpha_mm1), 0], \
                         [0, 0, 0, 1]])
    display(create_transormation_string(m-1, 'R'), T_mm1_R)

    T_R_P = sp.Matrix([[1, 0, 0, a_mm1], \
                       [0, 1, 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('R', 'P'), T_R_P)

    T_P_Q = sp.Matrix([[sp.cos(tm), -sp.sin(tm), 0, 0], \
                       [sp.sin(tm), sp.cos(tm), 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])   
    display(create_transormation_string('P', 'Q'), T_P_Q)

    T_Q_m = sp.Matrix([[1, 0, 0, translate_i], \
                       [0, 1, 0, translate_j], \
                       [0, 0, 1, translate_k], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('Q', m), T_Q_m)

    # Calculating the transformation matrix from frame m-1 to frame m
    T_mm1_m = sp.simplify(T_mm1_R * T_R_P * T_P_Q * T_Q_m)
    Individual_transformation_matrices.append(T_mm1_m)

    Full_forward_kinematics = sp.simplify(Full_forward_kinematics * T_mm1_m)

    display(create_transormation_string(m-1, m), T_mm1_m)

print("=== Full Forward Kinematics ===")
display(sp.simplify(Full_forward_kinematics))

print("=== Individual Transformation Matrices ===")
display(Individual_transformation_matrices)



=== LINK 1 ===


{}^{0}_RT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

{}^{R}_PT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

...

Matrix([
[1,                       0,                        0,                                                        -c_{boff}],
[0, cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), -c_{boc}*sin(pi*\beta_{cam}/180) + c_{h}*cos(pi*\beta_{cam}/180)],
[0, sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180),  c_{boc}*cos(pi*\beta_{cam}/180) + c_{h}*sin(pi*\beta_{cam}/180)],
[0,                       0,                        0,                                                                1]])

=== Full Forward Kinematics ===


Matrix([
[cos(theta_p)*cos(theta_y), -sin(theta_p)*cos(theta_y)*cos(pi*\beta_{cam}/180) + sin(theta_y)*sin(pi*\beta_{cam}/180), sin(theta_p)*sin(pi*\beta_{cam}/180)*cos(theta_y) + sin(theta_y)*cos(pi*\beta_{cam}/180), -c_{boff}*cos(theta_p)*cos(theta_y) + (a_b*cos(theta_p) - h_b*sin(theta_p))*cos(theta_y) + (c_{boc}*sin(pi*\beta_{cam}/180) - c_{h}*cos(pi*\beta_{cam}/180))*sin(theta_p)*cos(theta_y) + (c_{boc}*cos(pi*\beta_{cam}/180) + c_{h}*sin(pi*\beta_{cam}/180))*sin(theta_y)],
[sin(theta_y)*cos(theta_p), -sin(theta_p)*sin(theta_y)*cos(pi*\beta_{cam}/180) - sin(pi*\beta_{cam}/180)*cos(theta_y), sin(theta_p)*sin(theta_y)*sin(pi*\beta_{cam}/180) - cos(theta_y)*cos(pi*\beta_{cam}/180), -c_{boff}*sin(theta_y)*cos(theta_p) + (a_b*cos(theta_p) - h_b*sin(theta_p))*sin(theta_y) + (c_{boc}*sin(pi*\beta_{cam}/180) - c_{h}*cos(pi*\beta_{cam}/180))*sin(theta_p)*sin(theta_y) - (c_{boc}*cos(pi*\beta_{cam}/180) + c_{h}*sin(pi*\beta_{cam}/180))*cos(theta_y)],
[             sin(theta_p),              

=== Individual Transformation Matrices ===


[Matrix([
 [cos(theta_y), -sin(theta_y), 0,  0],
 [sin(theta_y),  cos(theta_y), 0,  0],
 [           0,             0, 1, d1],
 [           0,             0, 0,  1]]),
 Matrix([
 [cos(theta_p), -sin(theta_p),  0, 0],
 [           0,             0, -1, 0],
 [sin(theta_p),  cos(theta_p),  0, 0],
 [           0,             0,  0, 1]]),
 Matrix([
 [1, 0, 0, a_b],
 [0, 1, 0, h_b],
 [0, 0, 1,   0],
 [0, 0, 0,   1]]),
 Matrix([
 [1,                       0,                        0,                                                        -c_{boff}],
 [0, cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), -c_{boc}*sin(pi*\beta_{cam}/180) + c_{h}*cos(pi*\beta_{cam}/180)],
 [0, sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180),  c_{boc}*cos(pi*\beta_{cam}/180) + c_{h}*sin(pi*\beta_{cam}/180)],
 [0,                       0,                        0,                                                                1]])]

In [ ]:
# Creating a function

parameters = (d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg)

_var_d1 = 0.353
_var_theta_y = 0.0
_var_theta_p = 0.0
_var_a_b = 0.83
_var_h_b = 0.0761
_var_camera_beam_offset = 0.043
_var_camera_height = 0.029
_var_camera_offset_from_beam_centre = 0.04
_var_camera_angle_deg = 40

_ffk = sp.lambdify(parameters, Full_forward_kinematics, modules='numpy')

_fT01 = sp.lambdify(parameters, Individual_transformation_matrices[0], modules='numpy')
_fT12 = sp.lambdify(parameters, Individual_transformation_matrices[1], modules='numpy')
_fT23 = sp.lambdify(parameters, Individual_transformation_matrices[2], modules='numpy')
_fT34 = sp.lambdify(parameters, Individual_transformation_matrices[3], modules='numpy')

# Example calculation
T_cam = _ffk(
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

print("=== Full Forward Kinematics ===")
print(T_cam)

# Checking
print("=== Checking the individual transformation matrices ===")

_fkin_args = (
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

T01 = _fT01(*_fkin_args)
T12 = _fT12(*_fkin_args)
T23 = _fT23(*_fkin_args)
T34 = _fT34(*_fkin_args)

#Matrix multiplication
T_cam_check = T01 @ T12 @ T23 @ T34

print(T_cam_check)
matrices_correct = False

if np.allclose(T_cam, T_cam_check):
    print("The individual transformation matrices are correct.")
    matrices_correct = True


if matrices_correct:
    Base_coords = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    Frame_coords = [Base_coords]

    # Calculating the coordinates of each frame
    F1_coords = T01 @ Base_coords
    F2_coords = T01 @ T12 @ Base_coords
    F3_coords = T01 @ T12 @ T23 @ Base_coords
    Fcam_coords = T01 @ T12 @ T23 @ T34 @ Base_coords

    Frame_coords.append(F1_coords)
    Frame_coords.append(F2_coords)
    Frame_coords.append(F3_coords)
    Frame_coords.append(Fcam_coords)

# Plot frame origins and coordinate systems in 3D using Plotly
origins = np.array([frame[:3, 3] for frame in Frame_coords], dtype=float)
rotations = np.array([frame[:3, :3] for frame in Frame_coords], dtype=float)

fig = go.Figure()

# Plot frame origins
fig.add_trace(go.Scatter3d(
    x=origins[:, 0],
    y=origins[:, 1],
    z=origins[:, 2],
    mode='markers+lines',
    marker=dict(size=5),
    line=dict(color='black', width=2),
    name='Frame origins'
))

# Plot coordinate axes for each frame
axis_length = 0.1
axis_colors = ['red', 'green', 'blue']
axis_names = ['X axis', 'Y axis', 'Z axis']

for axis in range(3):
    x_values, y_values, z_values = [], [], []

    for origin, rotation in zip(origins, rotations):
        endpoint = origin + axis_length * rotation[:, axis]

        x_values.extend([origin[0], endpoint[0], None])
        y_values.extend([origin[1], endpoint[1], None])
        z_values.extend([origin[2], endpoint[2], None])

    fig.add_trace(go.Scatter3d(
        x=x_values,
        y=y_values,
        z=z_values,
        mode='lines',
        line=dict(color=axis_colors[axis], width=4),
        name=axis_names[axis]
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='3D Plot of Forward Kinematics'
)

fig.show()


=== Full Forward Kinematics ===
[[ 1.          0.          0.          0.787     ]
 [ 0.         -0.64278761 -0.76604444 -0.04928262]
 [ 0.          0.76604444 -0.64278761  0.42560378]
 [ 0.          0.          0.          1.        ]]
=== Checking the individual transformation matrices ===
[[ 1.          0.          0.          0.787     ]
 [ 0.         -0.64278761 -0.76604444 -0.04928262]
 [ 0.          0.76604444 -0.64278761  0.42560378]
 [ 0.          0.          0.          1.        ]]
The individual transformation matrices are correct.


In [6]:
import sys
import nbformat

print(sys.executable)
print(nbformat.__version__)
print(nbformat.__file__)

/Users/filipszczebak/Documents/Repositories/esa-ssp-tactile-legwheel/supporting_files/.venv/bin/python
5.11.1
/Users/filipszczebak/Documents/Repositories/esa-ssp-tactile-legwheel/supporting_files/.venv/lib/python3.12/site-packages/nbformat/__init__.py


: 

: 

: 

: 

: 

: 

: 

: 